# Phase 2 — run the scripts on Kaggle
Runs the three scripts of the repository in order and uploads both checkpoints to the Hugging Face Hub.

**Before *Run All*:** *Settings → Accelerator: GPU T4*, *Internet: On*; *Add Input* → the project dataset
(the repository folder); *Add-ons → Secrets* → `HF_TOKEN` (a Hugging Face **write** token), switched on for this notebook.
Full run ≈ 40 min.

In [1]:
import os, shutil, subprocess
from pathlib import Path
src = sorted(Path("/kaggle/input").glob("**/config.yaml"))[0].parent      # the uploaded repository
PROJECT = Path("/kaggle/working/abcd")
if not PROJECT.exists():
    shutil.copytree(src, PROJECT)
os.chdir(PROJECT)
print("project:", PROJECT)
!nvidia-smi --query-gpu=name --format=csv,noheader
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN found:", bool(os.environ["HF_TOKEN"]))

project: /kaggle/working/abcd
Tesla T4
Tesla T4
HF_TOKEN found: True


## 1. Continued pretraining (`train_cpt.py`, ≈ 30 min)

In [2]:
!python train_cpt.py --push

config.json: 100%|█████████████████████████████| 704/704 [00:00<00:00, 1.99MB/s]
tokenizer_config.json: 100%|███████████████| 3.66k/3.66k [00:00<00:00, 90.1kB/s]
vocab.json: 100%|████████████████████████████| 801k/801k [00:00<00:00, 2.95MB/s]
merges.txt: 100%|████████████████████████████| 466k/466k [00:00<00:00, 15.4MB/s]
special_tokens_map.json: 100%|█████████████████| 831/831 [00:00<00:00, 3.97MB/s]
tokenizer.json: 100%|██████████████████████| 2.10M/2.10M [00:00<00:00, 22.7MB/s]
downloading ABCD v1.1 from GitHub ...
10098 documents -> 10098 after removing 0 exact duplicates
                      split  documents  tokens
   train (unique documents)       8628 2565680
     train (policy docs x5)       8996 2658960
validation (early stopping)        502  150862
  held-out 10% (perplexity)       1010  302157
CPT data: 5193 training blocks x 512 tokens = 2,658,816 tokens per pass; 295 validation blocks; held-out 302,157 tokens

model.safetensors: downloading bytes: ███████████           |

## 2. SFT (`train_sft.py`, ≈ 2 min)

In [3]:
!python train_sft.py --push

overlap check: exact matches 0 | highest similarity 0.543 (threshold 0.6)
SFT: 533 train / 58 validation pairs; loss on 17,945 of 30,726 tokens (58%) = answers + <eos>
example (token -> in the loss?):
['Customer':-100] [' support':-100] [' conversation':-100] ['.':-100] ['\n':-100] ['Customer':-100] [':':-100] [' I':-100] ["'m":-100] [' a':-100] [' Silver':-100] [' member':-100] [' and':-100] [' my':-100] [' subscription':-100] [' payment':-100] [' is':-100] [' ':-100] ['1':-100] [' day':-100] [' late':-100] ['.':-100] [' Can':-100] [' I':-100] [' get':-100] [' an':-100] [' extension':-100] ['?':-100] ['\n':-100] ['Agent':-100] [':':-100] [' Yes':LOSS] ['.':LOSS] [' Silver':LOSS] [' members':LOSS] [' get':LOSS] [' an':LOSS] [' extension':LOSS] [' when':LOSS] [' they':LOSS] [' are':LOSS] [' only':LOSS] [' ':LOSS] ['1':LOSS] [' day':LOSS] [' late':LOSS] ['.':LOSS] ['<|endoftext|>':LOSS]
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 2645.69it/s]
{'loss': '2.238', 'gr

## 3. Score the three models on the 30 test questions (`score.py`, ≈ 5 min)

In [4]:
!python score.py
print(open('SCOREBOARD.md').read())


=== Small model, base  (HuggingFaceTB/SmolLM2-135M)
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 1442.90it/s]
[base] SP-01 score=0.0  (human) | 'Yes, please.'
[base] SP-02 score=0.0  (human) | 'Yes, I can book you a flight to New York for next week.'
[base] SP-03 score=0.0  (human) | '1 year.'
[base] SP-04 score=0.0  (human) | 'You can do that by clicking on the checkbox in the bottom right corner of the screen.'
[base] SP-05 score=0.0  (exact) | 'Yes.'
[base] SP-06 score=1.0  (exact) | 'Yes, you can return the product.'
[base] SP-07 score=0.0  (human) | 'You can choose from the following options:\n1. Cash Back\n2. Cash Back Plus\n3. Cash Back Plu'
[base] SP-08 score=1.0  (exact) | 'Yes.'
[base] SP-09 score=0.0  (exact) | '$30.'
[base] SP-10 score=0.0  (exact) | 'You can use the default amount.'
[base] SP-11 score=0.0  (human) | 'The return methods are described in the return.'
[base] SP-12 score=0.0  (human) | 'The refund status labels are in the support guidel

## 4. For the demo video
The same two test questions answered by the base, continued-pretrained and SFT models, then `score.py` with its results.

In [5]:
!python score.py --ask SP-07 SP-25


=== Small model, base  (HuggingFaceTB/SmolLM2-135M)
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 2482.92it/s]
[base] SP-07 score=0.0  (human) | 'You can choose from the following options:\n1. Cash Back\n2. Cash Back Plus\n3. Cash Back Plu'
[base] SP-25 score=0.0  (human) | 'Yes, they will.'

=== + continued pretraining  (/kaggle/working/abcd/checkpoints/cpt/best)
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 2150.55it/s]
[cpt] SP-07 score=0.0  (human) | 'Okay, I can help you with that.'
[cpt] SP-25 score=0.0  (human) | 'Yes, they will.'

=== + SFT  (/kaggle/working/abcd/checkpoints/sft/best)
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 2347.56it/s]
[sft] SP-07 score=1.0  (human) | 'The refund methods are by gift card, add value to the account, paper check or credit card.'
[sft] SP-25 score=0.0  (human) | 'Yes. The shipping fee is included with the purchase price.'

SP-07: When I start a refund, which refund methods ca

In [6]:
!python score.py --no-ppl


=== Small model, base  (HuggingFaceTB/SmolLM2-135M)
Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 2430.53it/s]
[base] SP-01 score=0.0  (human) | 'Yes, please.'
[base] SP-02 score=0.0  (human) | 'Yes, I can book you a flight to New York for next week.'
[base] SP-03 score=0.0  (human) | '1 year.'
[base] SP-04 score=0.0  (human) | 'You can do that by clicking on the checkbox in the bottom right corner of the screen.'
[base] SP-05 score=0.0  (exact) | 'Yes.'
[base] SP-06 score=1.0  (exact) | 'Yes, you can return the product.'
[base] SP-07 score=0.0  (human) | 'You can choose from the following options:\n1. Cash Back\n2. Cash Back Plus\n3. Cash Back Plu'
[base] SP-08 score=1.0  (exact) | 'Yes.'
[base] SP-09 score=0.0  (exact) | '$30.'
[base] SP-10 score=0.0  (exact) | 'You can use the default amount.'
[base] SP-11 score=0.0  (human) | 'The return methods are described in the return.'
[base] SP-12 score=0.0  (human) | 'The refund status labels are in the support guidel

## 5. Download the new results

In [7]:
import zipfile
with zipfile.ZipFile("/kaggle/working/scripts_run_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [Path("SCOREBOARD.md"), *Path("results").rglob("*"), *Path("figures").glob("*")]:
        if p.is_file():
            z.write(p)
print("download: /kaggle/working/scripts_run_results.zip")

download: /kaggle/working/scripts_run_results.zip
